# 04 — UncertaiNLP experiments
Fills every empirical \todo{} in **uncertainlp_certain.tex**: calibration, frontier, selective prediction.

## Inputs
`guard_scores_full.csv`. CPU only. Fills the UncertaiNLP paper: per-guard ρ **with CIs** (shared with Fig 3), **binned calibration plot**, **coverage–equality frontier LP**, **selective prediction curves**, and the recovery-percentage headline.

In [ ]:
import numpy as np, pandas as pd, math, json, itertools
from sklearn.metrics import cohen_kappa_score, roc_auc_score
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

SEED = 20260721
rng = np.random.default_rng(SEED)
DICES_URL = ("https://raw.githubusercontent.com/google-research-datasets/"
             "dices-dataset/main/350/diverse_safety_adversarial_dialog_350.csv")
df = pd.read_csv(DICES_URL)
df["unsafe"] = (df.Q_overall == "Yes").astype(int)
scores = pd.read_csv("guard_scores_full.csv")          # from notebook 01
GUARDS = [c for c in scores.columns if c.startswith("p_")]
S = scores.set_index("item_id")

pooled = df.groupby("item_id")["unsafe"].mean()
pc = np.clip(pooled, 1e-9, 1-1e-9)
entropy = -(pc*np.log2(pc) + (1-pc)*np.log2(1-pc))
MIN_RATERS = 10
counts = df.groupby(["item_id","rater_race"])["unsafe"].count().unstack()
means  = df.groupby(["item_id","rater_race"])["unsafe"].mean().unstack()
race = (means > .5).astype(float).where(counts >= MIN_RATERS)
contested = race.index[race.nunique(axis=1) > 1]
Q_UNSAFE = 1 - pooled.mean()   # base-rate-matched threshold quantile
def thr_of(col): return np.nanquantile(S[col], Q_UNSAFE)
print(f"{len(GUARDS)} guards | {len(contested)} contested items | pooled rate {pooled.mean():.3f}")

## Binned calibration — confidence by rater-entropy decile

In [ ]:
fig, ax = plt.subplots(figsize=(4.5,3))
for col in GUARDS:
    s = S[col].loc[entropy.index]; ok = s.notna()
    conf = (s[ok]-thr_of(col)).abs()
    H = entropy[s.index[ok]]
    dec = pd.qcut(H, 10, labels=False, duplicates="drop")
    prof = conf.groupby(dec).mean()
    ax.plot(prof.index, prof.values, marker="o", ms=3, label=col.replace("p_",""))
ax.set_xlabel("rater-entropy decile (0=consensus, 9=max disagreement)")
ax.set_ylabel("mean guard confidence")
ax.legend(fontsize=7); ax.set_title("Disagreement-calibrated confidence should slope DOWN", fontsize=9)
plt.tight_layout(); plt.savefig("fig_calibration_bins.pdf"); plt.show()

## Coverage–equality frontier (model-free LP)
For each coverage level: choose per-pattern positives $t_p$ and abstentions $a_p$ to maximize minimum-group agreement on covered items. Linear because the covered count is fixed per coverage level.

In [ ]:
from scipy.optimize import linprog
from collections import Counter
pats_ser = race.dropna().apply(lambda r: tuple(int(x) for x in r), axis=1)
pcnt = Counter(pats_ser); pats = list(pcnt.keys())
n_p = np.array([pcnt[p] for p in pats], float); P = np.array(pats, float)
K = P.shape[1]; N = n_p.sum(); m = len(pats)

def frontier_point(coverage):
    # vars: t_p (positives), a_p (abstain), z ; covered = N*coverage (equality constraint)
    # agree_d = sum_p [ t_p*p_d + (n_p - t_p - a_p)*(1-p_d) ]  >= z * N * coverage
    nv = 2*m + 1
    c = np.zeros(nv); c[-1] = -1
    A_ub, b_ub = [], []
    for d in range(K):
        row = np.zeros(nv)
        row[:m]      = -(2*P[:,d] - 1)          # -t coefficient
        row[m:2*m]   = (1 - P[:,d])             # +a coefficient (abstain removes negatives' credit)
        row[-1]      = N * coverage
        A_ub.append(row); b_ub.append((n_p * (1 - P[:,d])).sum())
    A_eq = [np.concatenate([np.zeros(m), np.ones(m), [0]])]
    b_eq = [N * (1 - coverage)]
    ub = []
    for i in range(m): ub.append((0, n_p[i]))
    for i in range(m): ub.append((0, n_p[i]))
    ub.append((None, None))
    # also need t_p + a_p <= n_p
    for i in range(m):
        row = np.zeros(nv); row[i] = 1; row[m+i] = 1
        A_ub.append(row); b_ub.append(n_p[i])
    r = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                A_eq=np.array(A_eq), b_eq=np.array(b_eq), bounds=ub, method="highs")
    return -r.fun if r.success else np.nan

covs = np.linspace(.70, 1.0, 16)
front = [frontier_point(cv) for cv in covs]
plt.figure(figsize=(4.5,3))
plt.plot(covs, front, "k-", lw=2, label="LP frontier (any classifier)")
plt.xlabel("coverage"); plt.ylabel("max-min group agreement (covered items)")
print("anchors: full-coverage", round(front[-1],3), "| 78% coverage", round(frontier_point(.78),3))

## Selective prediction from guard scores — how much of the frontier is reachable?
Two abstention policies per guard: (a) score-band around τ; (b) entropy predictor (cross-validated logistic on all guards' scores). Recovery % = share of the ideal equality gain achieved at fixed coverage.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

def min_group_agree(cov_mask, verdict):
    vals = []
    for grp in race.columns:
        gm = race[grp].loc[cov_mask.index]
        m = cov_mask & gm.notna() & verdict.notna()
        if m.sum() >= 20: vals.append(float((gm[m] == verdict[m]).mean()))
    return min(vals) if len(vals) >= 2 else np.nan

X = S[GUARDS].loc[entropy.index]
okX = X.notna().all(axis=1)
hiH = (entropy[okX.index[okX]] > entropy.median()).astype(int)
pred_hiH = pd.Series(
    cross_val_predict(LogisticRegression(max_iter=1000), X[okX], hiH,
                      cv=5, method="predict_proba")[:,1], index=okX.index[okX])

for col in GUARDS:
    s = S[col].loc[entropy.index]; t = thr_of(col)
    v = (s >= t).astype(float); v[s.isna()] = np.nan
    base = min_group_agree(s.notna(), v)
    print(f"\n== {col.replace('p_','')} (full-coverage min-group agree = {base:.3f}) ==")
    for target_cov in (.9, .8):
        # (a) score-band abstention
        band = (s - t).abs().rank(pct=True)
        keep_a = s.notna() & (band > (1 - target_cov))
        ma = min_group_agree(keep_a, v)
        # (b) entropy-predictor abstention
        pr = pred_hiH.reindex(s.index)
        keep_b = s.notna() & pr.notna() & (pr.rank(pct=True) < target_cov)
        mb = min_group_agree(keep_b, v)
        ceiling = frontier_point(target_cov)
        rec = lambda x: 100*(x-base)/(ceiling-base) if (not math.isnan(x)) and ceiling>base else float('nan')
        print(f"  cov={target_cov:.0%}: band={ma:.3f} ({rec(ma):.0f}% of ideal) | "
              f"entropy-pred={mb:.3f} ({rec(mb):.0f}% of ideal) | frontier={ceiling:.3f}")
print("\nHeadline for the paper: the recovery %s at 80/90% coverage; expect band << entropy-pred, matching the calibration finding.")

## Export — paste-ready numbers
Dump every value the UncertaiNLP tex needs into one JSON; fill \todo{}s only from this file.

In [ ]:
out = {"note": "fill uncertainlp_certain.tex ONLY from these values", "seed": SEED}
json.dump(out, open("uncertainlp_numbers.json","w"), indent=1)
print("Extend the dict above as cells run; keep one source of truth per paper.")